# Loan Approval Prediction — Data Cleaning & Feature Engineering

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np

from src.preprocessing import (
    load_raw_data, drop_identifier_columns, fix_negative_asset_values,
    remove_duplicate_rows, encode_categorical_columns, clean_data
)
from src.feature_engineering import engineer_features
from src.utils import print_section

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1. Load raw data and confirm the negative-value issue

In [2]:
raw = load_raw_data("../data/loan_approval_dataset.csv")

print_section("Before cleaning")
print(f"Shape: {raw.shape}")
print(f"Negative residential_assets_value rows: {(raw['residential_assets_value'] < 0).sum()}")
raw[raw['residential_assets_value'] < 0][['residential_assets_value']]



Before cleaning
Shape: (4269, 13)
Negative residential_assets_value rows: 28


,residential_assets_value
59,-100000
196,-100000
559,-100000
702,-100000
737,-100000
784,-100000
904,-100000
1089,-100000
1163,-100000
1350,-100000


## 2. Run the cleaning pipeline

In [3]:
df = clean_data("../data/loan_approval_dataset.csv")

print_section("After cleaning")
print(f"Shape: {df.shape}")
print(f"Negative residential_assets_value rows: {(df['residential_assets_value'] < 0).sum()}")
print(f"Columns: {list(df.columns)}")
df.head()



After cleaning
Shape: (4269, 12)
Negative residential_assets_value rows: 0
Columns: ['no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']


,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,2,1,0,9600000,29900000,12,778,2400000,17600000,22700000,8000000,1
1,0,0,1,4100000,12200000,8,417,2700000,2200000,8800000,3300000,0
2,3,1,0,9100000,29700000,20,506,7100000,4500000,33300000,12800000,0
3,3,1,0,8200000,30700000,8,467,18200000,3300000,23300000,7900000,0
4,5,0,1,9800000,24200000,20,382,12400000,8200000,29400000,5000000,0


**What `clean_data()` just did (see `src/preprocessing.py` for the full implementation):**
1. Loaded the raw CSV, stripped whitespace from column names and string values.
2. Dropped `loan_id` (pure identifier, no predictive value).
3. Clipped negative values in the four asset columns to 0.
4. Removed exact duplicate rows (none were found, but this step is kept for robustness against future data refreshes).
5. Encoded `education` (Graduate=1/Not Graduate=0), `self_employed` (Yes=1/No=0), and the target `loan_status` (Approved=1/Rejected=0) with fixed, deterministic mappings.


## 3. Feature engineering

In [4]:
df = engineer_features(df)

print_section("New features added")
new_cols = ["total_assets_value", "loan_to_income_ratio", "asset_to_loan_ratio", "cibil_band"]
df[new_cols].head()



New features added


,total_assets_value,loan_to_income_ratio,asset_to_loan_ratio,cibil_band
0,50700000,3.114583,1.695652,Good
1,17000000,2.975610,1.393443,Poor
2,57700000,3.263736,1.942761,Poor
3,52700000,3.743902,1.716612,Poor
4,55000000,2.469388,2.272727,Poor


**New features (see `src/feature_engineering.py`):**
- `total_assets_value` — sum of all four asset columns. A single combined signal, and reduces multicollinearity risk from four correlated source columns going into modeling.
- `loan_to_income_ratio` — `loan_amount / income_annum`. A standard credit-risk signal: how large the request is relative to earning capacity.
- `asset_to_loan_ratio` — `total_assets_value / loan_amount`. How well the applicant's assets cover the requested loan.
- `cibil_band` — `cibil_score` bucketed into Poor / Fair / Good / Excellent. Kept alongside the raw score so simpler models can exploit threshold effects that tree-based models pick up automatically.


## 4. Sanity checks on the engineered features

In [5]:
print_section("Descriptive stats — new numeric features")
df[["total_assets_value", "loan_to_income_ratio", "asset_to_loan_ratio"]].describe()



Descriptive stats — new numeric features


,total_assets_value,loan_to_income_ratio,asset_to_loan_ratio
count,4.269000e+03,4269.000000,4269.000000
mean,3.254943e+07,2.984807,2.232165
std,1.950594e+07,0.595496,0.642788
min,5.000000e+05,1.500000,0.750000
25%,1.630000e+07,2.464286,1.767347
50%,3.150000e+07,3.000000,2.142857
75%,4.720000e+07,3.500000,2.616216
max,9.070000e+07,4.000000,5.666667


In [6]:
print_section("cibil_band distribution")
df["cibil_band"].value_counts()



cibil_band distribution


cibil_band
Poor         2468
Fair          745
Excellent     695
Good          361
Name: count, dtype: int64

## 5. Confirm no missing values were introduced

In [7]:
missing = df.isnull().sum()
missing[missing > 0] if missing.sum() > 0 else print("No missing values.")


No missing values.


## 6. Save the cleaned + engineered dataset

In [8]:
OUTPUT_PATH = "../data/loan_approval_cleaned.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset: {OUTPUT_PATH}")
print(f"Final shape: {df.shape}")


Saved cleaned dataset: ../data/loan_approval_cleaned.csv
Final shape: (4269, 16)


## 7. Summary

- The dataset was successfully cleaned and prepared for analysis. The unnecessary `loan_id` column was removed, and four new features were created through feature engineering to provide additional information for model development.

- Negative values found in the asset-related columns were corrected by replacing them with zero, ensuring that all asset values remained valid and consistent.

- Categorical features, including `education`, `self_employed`, and the target variable `loan_status`, were converted into numerical values using consistent binary encoding, making them suitable for machine learning algorithms.

- Four new features were introduced: `total_assets_value`, `loan_to_income_ratio`, `asset_to_loan_ratio`, and `cibil_band`. These engineered features were designed to better represent an applicant's financial profile and improve the model's predictive capability.

- Feature scaling was intentionally postponed until after the dataset was divided into training and testing sets. This prevents **data leakage**, ensuring that information from the test data does not influence the training process and allowing for a fair evaluation of model performance.

- The processed dataset is now ready for exploratory data analysis, including univariate, bivariate, and multivariate analysis, followed by correlation analysis and feature selection before training the machine learning models.
